## Data Preparation Notebook

### Prerequisites
1. Run `scripts/setup.sql` first to create the Snowflake infrastructure and Git integration

### Dataset Information
This quickstart uses the **DeepPCB** dataset - a public open-source dataset for PCB defect detection.
- **Source:** [Surface-Defect-Detection](https://github.com/Charmve/Surface-Defect-Detection/tree/master/DeepPCB)
- **License:** Please review and comply with the dataset's licensing terms before use.

### How it works
This notebook reads data directly from the **Git repository stage** - no manual upload needed!
The PCB dataset is included in the repo under `data/PCBData/` and accessed via:
```
@PCB_GITHUB_REPO/branches/main/data/PCBData/
```

### Purpose
This notebook processes images and labels from the Git repo, converts them to base64, and creates the training data tables.
 

## Step 1: Import Libraries
Import the necessary Python packages for data processing.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import json
import os
import base64

## Step 2: Connect to Snowflake
Establish a connection to Snowflake and verify the session parameters.

In [ ]:
# Get Snowflake Session object
session = get_active_session()
session.sql_simplifier_enabled = True

# Add a query tag to the session
session.query_tag = {"origin":"sf_sit-is", 
                     "name":"distributed_ml_crt_imageanomaly_detection", 
                     "version":{"major":1, "minor":0},
                     "attributes":{"is_quickstart":1, "source":"notebook"}}

# Current Environment Details
print('Connection Established with the following parameters:')
print('User      : {}'.format(session.get_current_user()))
print('Role      : {}'.format(session.get_current_role()))
print('Database  : {}'.format(session.get_current_database()))
print('Schema    : {}'.format(session.get_current_schema()))
print('Warehouse : {}'.format(session.get_current_warehouse()))

## Step 3: Load Data from Git Repository
Fetch the latest data from the Git repository and copy it to DATA_STAGE for processing.

The Git repository contains PCB images and label files. We'll copy them to DATA_STAGE for easier processing.

In [ ]:
# Fetch latest from Git repository
session.sql("ALTER GIT REPOSITORY PCB_GITHUB_REPO FETCH").collect()

# Copy data from Git repo to DATA_STAGE for processing
print("\nCopying data from Git repo to DATA_STAGE...")
session.sql("COPY FILES INTO @DATA_STAGE/labels/ FROM @PCB_GITHUB_REPO/branches/main/data/PCBData/ PATTERN='.*_not/.*\\.txt'").collect()
session.sql("COPY FILES INTO @DATA_STAGE/images/ FROM @PCB_GITHUB_REPO/branches/main/data/PCBData/ PATTERN='.*_test\\.jpg'").collect()
print("Data copied to DATA_STAGE!")

# Verify
print("\nLabels in DATA_STAGE:")
session.sql("LIST @DATA_STAGE/labels/").show()
print("\nImages in DATA_STAGE:")
session.sql("LIST @DATA_STAGE/images/").show()



## Step 4: Process Label Files
Download and parse label files from DATA_STAGE. Each label file contains bounding box coordinates (xmin, ymin, xmax, ymax) and defect class IDs:
- **0**: open
- **1**: short  
- **2**: mousebite
- **3**: spur
- **4**: copper
- **5**: pin-hole

In [ ]:
import os
import base64
import pandas as pd

# Download labels from DATA_STAGE
local_train_dir = "/tmp/labels/"
os.makedirs(local_train_dir, exist_ok=True)

# Download all label files
print("Downloading label files...")
session.file.get("@DATA_STAGE/labels/", local_train_dir)
print(f"Downloaded labels to {local_train_dir}")

path_annot = "/tmp/labels/"

# Initialize a list to hold all the data  
data = []  
  
# Walking through the directory to get all label files  
for path, subdirs, files in os.walk(path_annot):  
    for name in files:  
        if name.endswith('.txt'):  # Filter to include only .txt files  
            full_path = os.path.join(path, name)  
            with open(full_path, 'r') as file:  
                for line in file:  
                    parts = line.strip().split()  
                    if len(parts) == 5:
                        xmin, ymin, xmax, ymax,class_id = parts  
                        data.append({  
                            "filename": name.replace('.txt', ''), 
                            "xmin": float(xmin),  
                            "ymin": float(ymin),  
                            "xmax": float(xmax),  
                            "ymax": float(ymax) ,
                            "class": int(class_id)
                        })  
  
# Create a DataFrame  
trainlabels_df = pd.DataFrame(data)
print(f"Parsed {len(trainlabels_df)} label entries")

# Table already created by setup.sql - truncate and insert
session.sql("TRUNCATE TABLE LABELS_TRAIN").collect()
session.write_pandas(trainlabels_df, "LABELS_TRAIN", overwrite=False, quote_identifiers=False)
print("Labels saved to LABELS_TRAIN table.")

## Step 5: Process Images and Create Training Dataset
Download PCB images, encode them to Base64, and merge with label data to create the `TRAIN_IMAGES_LABELS` table.

In [ ]:
import os
import base64
import pandas as pd

# Download images from DATA_STAGE
local_train_images_dir = "/tmp/images/train"
os.makedirs(local_train_images_dir, exist_ok=True)

# Download all image files
print("Downloading image files...")
session.file.get("@DATA_STAGE/images/", local_train_images_dir)
print(f"Downloaded images to {local_train_images_dir}")

# Convert downloaded images to Base64
images_data = []
for filename in os.listdir(local_train_images_dir):
    if filename.endswith("_test.jpg"):
        filepath = os.path.join(local_train_images_dir, filename)
        with open(filepath, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
            images_data.append({"Filename": filename.replace(".jpg", ""), "image_data": encoded_string})

train_images_base64_data = images_data
print(f"Processed {len(images_data)} images")

# Convert to DataFrames for inserting into separate tables
train_images_df = pd.DataFrame(train_images_base64_data)

print("trainlabels_df columns:", trainlabels_df.columns)
print("train_images_df columns:", train_images_df.columns)

if 'filename' in trainlabels_df.columns:
    trainlabels_df.rename(columns={'filename': 'Filename'}, inplace=True)
# Update labels DataFrame to include `_test.jpg` suffix for matching
trainlabels_df['Filename'] = trainlabels_df['Filename'] + "_test"

merged_train_df = pd.merge(trainlabels_df, train_images_df, how='inner', on='Filename')



# Table already created by setup.sql - just truncate and insert
session.sql("TRUNCATE TABLE TRAIN_IMAGES_LABELS").collect()

# Insert merged data using write_pandas (faster than row-by-row insert)
merged_train_df['Filename'] = merged_train_df['Filename'].str.replace('_test', '')
session.write_pandas(merged_train_df, "TRAIN_IMAGES_LABELS", overwrite=False, quote_identifiers=False)

print(f"Inserted {len(merged_train_df)} rows into TRAIN_IMAGES_LABELS table.")


trainlabels_df columns: Index(['filename', 'xmin', 'ymin', 'xmax', 'ymax', 'class'], dtype='object')
train_images_df columns: Index(['Filename', 'image_data'], dtype='object')






## Step 6: Split Data into Training and Test Sets
Split the dataset into 90% training and 10% test data for model evaluation.


In [ ]:
from sklearn.model_selection import train_test_split

df = session.table("TRAIN_IMAGES_LABELS").to_pandas()
print(f"Total samples: {len(df)}")

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Training samples: {len(train_df)}, Test samples: {len(test_df)}")

# Create training_data and test_data tables
session.sql("CREATE TABLE IF NOT EXISTS training_data AS SELECT * FROM TRAIN_IMAGES_LABELS WHERE 1=0").collect()
session.sql("CREATE TABLE IF NOT EXISTS test_data AS SELECT * FROM TRAIN_IMAGES_LABELS WHERE 1=0").collect()

session.sql("TRUNCATE TABLE training_data").collect()
session.sql("TRUNCATE TABLE test_data").collect()

session.write_pandas(train_df, "training_data", overwrite=False, quote_identifiers=False)
session.write_pandas(test_df, "test_data", overwrite=False, quote_identifiers=False)

print("Data split complete!")


## ✅ Data Preparation Complete!

**Tables created:**
- `LABELS_TRAIN` - Parsed label data with bounding box coordinates
- `TRAIN_IMAGES_LABELS` - Combined images (base64) with labels
- `training_data` - 90% of data for model training
- `test_data` - 10% of data for model evaluation

**Next Step:** Run `1_Distributed_Model_Training_Snowflake_Notebooks.ipynb` to train the defect detection model.